# SSLimPy — Halo Model and Line-emission Astrophysics

This notebook inspects the halo-model ingredients (mass function, bias,
concentration, halo-model power spectrum) and the astrophysical ingredients
(mass–luminosity/luminosity-function prescriptions, line averaged brightness
temperature and bias) of a SSLimPy run.

In [ ]:
import os
import sys

sys.path.append("../")

# Use the available cores for the Einstein--Boltzmann solver (CAMB/CLASS) and numba
os.environ.setdefault("OMP_NUM_THREADS", "12")

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

Cs = sns.color_palette("colorblind")

import niceplots
niceplots.utils.initPlot()

## 1. Settings, cosmology and halo model

In [ ]:
settings = {
    "code": "camb",
    "do_RSD": True,
    "nonlinearRSD": True,
    "QNLpowerspectrum": True,
    "halo_model_PS": True,        # halo-model shot noise needs an actual halo model
    "output": ["Power spectrum"],
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 10 * u.Mpc**-1,
    "nk": 50,
}

cosmodict = {
    "h": 0.677,
    "Omegam": 0.309177,
    "Omegab": 0.04903,
    "sigma8": 0.8222,
    "ns": 0.96824,
    "mnu": 0.06,
    "Neff": 3.044,
}

halodict = {
    "halo_tracer": "clustering",   # compute halo quantities from the CDM+baryon field
    "hmf_model": "ST",             # Sheth--Tormen mass function
    "concentration": "Diemer19",   # Diemer & Joyce (2019)
    "bias_model": "ST99",          # Sheth--Tormen (1999)
    "bias_par": {"q": 0.707, "p": 0.3},
}

In [ ]:
from SSLimPy.interface import sslimpy

myssl = sslimpy.SSLimPy(settings_dict=settings, cosmopars=cosmodict, halopars=halodict)

halo = myssl.current_halomodel
cosmo = halo.cosmology

## 2. Halo mass function, bias and concentration

In [ ]:
M = np.geomspace(1e9, 1e15, 100) * u.Msun
z = 0.5

dndM = halo.halomassfunction(M, z)
b1 = halo.halobias(M, z)
c = halo.concentration(M, z)

fw, fh = plt.rcParams['figure.figsize']
fig, axs = plt.subplots(1, 3, figsize=(2 * fw, 1.2 * fh))

color = iter(Cs)
axs[0].loglog(M, M * dndM, c=next(color))
axs[0].set_xlabel(r"$M\,[M_\odot]$")
axs[0].set_ylabel(r"$M\,\mathrm{d}n/\mathrm{d}M\,[\mathrm{Mpc}^{-3}]$")
axs[1].semilogx(M, b1, c=next(color))
axs[1].set_xlabel(r"$M\,[M_\odot]$")
axs[1].set_ylabel(r"$b_1(M)$ at $z=0.5$")
axs[2].semilogx(M, c, c=next(color))
axs[2].set_xlabel(r"$M\,[M_\odot]$")
axs[2].set_ylabel(r"$c(M)$ at $z=0.5$")
plt.tight_layout()

## 3. Halo-model power spectrum

`P_halo(k, z)` evaluates the one-halo (plus off-centring) and two-halo terms and
combines them. The cosmological shot noise follows from the same halo model when
`halo_model_PS = True`.

In [ ]:
k = np.geomspace(1e-3, 10, 100) * u.Mpc**-1

Plin = cosmo.matpow(k, z, tracer="clustering", nonlinear=False)
Phalo = halo.P_halo(k, z)

plt.loglog(k, Plin, c=Cs[0], label="linear $P_{cc}(k)$")
plt.loglog(k, np.squeeze(Phalo), c=Cs[1], ls="--", label="halo model")
plt.xlabel(r"$k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"$P(k)\,[\mathrm{Mpc}^{3}]$")
plt.legend()

### Available halo-model prescriptions

* `hmf_model`: `PS`, `ST`, `Tinker`, `Crocce`, `Jenkins`, `Warren`, `Watson`,
  `Watson_FOF`, `Angulo`.
* `bias_model`: `ST99`, `Tinker10`, `Tinker05`, `Mo96`, `Jing98`, `SMT01`,
  `Seljak04`, `Seljak04_Cosmo`, `Mandelbaum05`.
* `concentration`: `Diemer19`, `Bullock`, `Duffy`.
* `nonlinear_bias` (for the Trispectrum): `HMF` (b1_HMF, b2sph_HMF, b3_HMF),
  `lazeyras` (b2sph_lazeyras, b3_lazeyras), or `fitted` (b2/b3 through the
  Peacock & Dodds fit).

## 4. Line-emission astrophysics

The `AstroFunctions` object holds the line prescription (mass–luminosity
relation `ML` or luminosity function `LF`) and the survey specifications, and
derives the astrophysical functions used in the power spectrum:
line-averaged temperature $\bar T$, bias, `CLT(z)` normalising
$L(M,z) \to \delta_T(M,z)$, etc.

In [ ]:
from SSLimPy.interface import updater

astrodict_CII = {
    "model_type": "ML",                    # mass--luminosity relation
    "model_name": "SilvaCII",              # Silva et al. (2015) [CII] model
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter": 0.37,
    "meanperserve_scatter": True,
}

surveyspecs_CII = {
    "Tsys_NEFD": 40 * u.uK,        # instrument: system temperature (NEFD)
    "Nfeeds": 19,
    "nD": 1,
    "beam_FWHM": 0.5 * u.arcmin,
    "nu": 1.897 * u.THz,           # rest-frame frequency of [CII]
    "nuObs": 250 * u.GHz,          # observed frequency   -> z ~ 6.6
    "Delta_nu": 50 * u.GHz,        # width of each frequency bin
    "dnu": 900 * u.MHz,            # spectral resolution
    "tobs": 1300 * u.h,
    "Omega_field": 140 * u.deg**2,
}

myastro = updater.update_astro(
    myssl.current_astro,
    cosmo.cosmopars, halo.haloparams,
    astrodict_CII, surveyspecs_CII, myssl.cfg,
)

$\bar T_{\rm line}(z)$ and the luminosity-weighted line bias
$\langle b \rangle$ (linear, from the halo model) are the two quantities that
enter the monopole of the observed power spectrum.

In [ ]:
z = np.linspace(0, 6, 100)

Tav = myastro.Tavg(z)
b_halo = halo.bavg(1, z, 1)
b_line = myastro.bavg(1, z, 1)
fw, fh = plt.rcParams['figure.figsize']
fig, axs = plt.subplots(1, 2, figsize=(2 * fw, 1.5 * fh))
axs[0].plot(z, Tav)
axs[0].set_xlabel(r"$z$")
axs[0].set_ylabel(r"$\bar{T}_{\rm CII}(z)\,[\mathrm{uK}]$")
axs[1].plot(z, b_halo, c=Cs[0], label=r"$\langle b\rangle_{\rm halo}$")
axs[1].plot(z, b_line, c=Cs[1], ls="--", label=r"$b_{\rm line}$")
axs[1].set_xlabel(r"$z$")
axs[1].set_ylabel("bias")
axs[1].legend()
plt.tight_layout()

The mass–luminosity relation $L(M,z)$ at a given redshift:

In [ ]:
Mtest = np.geomspace(1e8, 1e13, 100) * u.Msun
ztest = 2.0

L = myastro.massluminosityfunction(Mtest, ztest)

plt.loglog(Mtest, L.to(u.Lsun))
plt.xlabel(r"$M\,[M_\odot]$")
plt.ylabel(r"$L_{\rm CII}(M)\,[L_\odot]$ at $z=2$")

### Available line prescriptions

* `model_type = "ML"` (`mass_luminosity.py`): `MassPow`, `DblPwr`,
  `CO_lines_scaling_LFIR`, `COMAP_Fid`, `TonyLi`, `SilvaCII`,
  `FonsecaLyalpha`, `SilvaLyalpha_12`, `Chung_Lyalpha`, `KSrel`,
  `GongHalpha`, `GongHbeta`, `GongOIII`, `GongOII`,
  `HI_lowz_Villaescusa`, `L_from_MHI_VN`, `MHI_21cm_Obuljen`,
  `MHI_21cm_Padmanabhan`, `Constant_L`.
* `model_type = "LF"` (`luminosity_functions.py`): `Sch`, `SchCut`.

Each model documents its `model_par` parameters in
`SSLimPy/cosmology/fitting_functions/`. SFR files referenced by
`SFR_file` are read from `SFR_tables/`.